# ✈️ Multi-Agent AI Travel Planner
### LangGraph · Groq LLaMA 3.1-8B-Instant · Serper API · Gradio UI

**7-Agent Architecture:**
- 🔍 Research Agent, ✈️ Flight Agent, 🏨 Hotel Agent, 🌤️ Weather Agent
- 💰 Budget Agent, 🗺️ Itinerary Agent, 📋 Booking Agent

In [11]:
## CELL 1: Install packages
get_ipython().system('pip install -q langgraph langchain langchain-groq langchain-community')
get_ipython().system('pip install -q "gradio>=4.0.0" requests python-dotenv pydantic')
print("Packages installed!")

Packages installed!


In [12]:
## CELL 2: API Keys
import os
GROQ_API_KEY   = "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"
SERPER_API_KEY = "34cf9bb4a6965f4ad9bcce86f48f5d693048ac09"
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY
print(f"Keys set! Groq: ...{GROQ_API_KEY[-4:]}")

Keys set! Groq: ...yxAT


In [13]:
## CELL 3: Imports and LLM
import os, requests
from datetime import datetime, timedelta
from typing import TypedDict, Annotated, List
import operator
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.7,
    max_tokens=2048,
)
print(f"LLM ready: {llm.model_name}")

LLM ready: llama-3.1-8b-instant


In [14]:
## CELL 4: Serper Search
def serper_search(query, num=5):
    url = "https://google.serper.dev/search"
    headers = {"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"}
    try:
        r = requests.post(url, headers=headers, json={"q": query, "num": num}, timeout=12)
        r.raise_for_status()
        d = r.json()
        out = []
        kg = d.get("knowledgeGraph", {})
        if kg:
            out.append(f"Featured: {kg.get('title','')} - {kg.get('description','')}")
        for x in d.get("organic", [])[:num]:
            out.append(f"* {x.get('title','')}\n  {x.get('snippet','')}\n  {x.get('link','')}")
        return "\n\n".join(out) if out else "No results."
    except Exception as e:
        return f"[Search unavailable: {e}]"

t = serper_search("Paris travel", num=1)
print("Serper:", "OK" if "unavailable" not in t else "Fallback")

Serper: OK


In [15]:
## CELL 5: TravelState + All 7 Agents

class TravelState(TypedDict):
    origin: str; destination: str; travel_type: str
    departure_date: str; return_date: str; num_travelers: int
    budget_level: str; budget_amount: float; currency: str
    interests: List[str]; accommodation_type: str; flight_class: str
    special_requirements: str; destination_research: str
    flight_options: str; hotel_options: str; weather_info: str
    budget_analysis: str; itinerary: str; booking_checklist: str
    visa_info: str; messages: Annotated[List, operator.add]
    current_agent: str; errors: List[str]; progress: int

def call_llm(sys, usr):
    try:
        return llm.invoke([SystemMessage(content=sys), HumanMessage(content=usr)]).content
    except Exception as e:
        return f"[Error: {e}]"

def destination_research_agent(s):
    print("Researching destination...")
    dest = s["destination"]; t = s["travel_type"]
    ints = ", ".join(s.get("interests", []))
    sr   = serper_search(f"travel guide {dest} attractions {t} {ints}", 4)
    visa = serper_search(f"visa requirements {dest}", 2)
    res  = call_llm(
        "Expert travel researcher. Write overview: attractions, culture, areas, transport, cuisine, safety. Use emojis.",
        f"Destination: {dest} | Type: {t} | Interests: {ints}\nSearch:\n{sr}\nVisa:\n{visa}\nWrite research report."
    )
    return {**s, "destination_research": res, "visa_info": visa,
            "current_agent": "flight", "progress": 15,
            "messages": [AIMessage(content=f"Research done: {dest}")]}

def flight_search_agent(s):
    print("Searching flights...")
    sr = serper_search(f"flights {s['origin']} to {s['destination']} {s['departure_date']} {s['flight_class']} best price", 5)
    res = call_llm(
        "Flight analyst. Top 3 airlines with prices, booking tips, direct vs layover, baggage, total cost.",
        f"Route: {s['origin']} to {s['destination']} | {s['departure_date']} - {s['return_date']}\nTravelers: {s['num_travelers']} | Class: {s['flight_class']} | Budget: {s['currency']} {s['budget_amount']}\nSearch:\n{sr}\nFlight recommendations."
    )
    return {**s, "flight_options": res, "current_agent": "hotel", "progress": 30,
            "messages": [AIMessage(content="Flights done")]}

def hotel_search_agent(s):
    print("Searching hotels...")
    sr = serper_search(f"best {s['budget_level']} {s['accommodation_type']} {s['destination']} reviews", 5)
    res = call_llm(
        "Hotel consultant. Top 4 properties with star ratings, price/night, location, amenities, booking tips.",
        f"Destination: {s['destination']} | {s['departure_date']} - {s['return_date']}\nGuests: {s['num_travelers']} | Type: {s['accommodation_type']} | Budget: {s['budget_level']}\nSearch:\n{sr}\nHotel recommendations."
    )
    return {**s, "hotel_options": res, "current_agent": "weather", "progress": 45,
            "messages": [AIMessage(content="Hotels done")]}

def weather_agent(s):
    print("Checking weather...")
    sr = serper_search(f"weather forecast {s['destination']} {s['departure_date']} temperature", 4)
    res = call_llm(
        "Travel meteorologist. Conditions, temp range, precipitation, packing advice, activity adjustments. Use weather emojis.",
        f"Destination: {s['destination']} | Dates: {s['departure_date']} to {s['return_date']}\nSearch:\n{sr}\nWeather analysis."
    )
    return {**s, "weather_info": res, "current_agent": "budget", "progress": 58,
            "messages": [AIMessage(content="Weather done")]}

def budget_agent(s):
    print("Analyzing budget...")
    try:
        days = (datetime.strptime(s["return_date"], "%Y-%m-%d") - datetime.strptime(s["departure_date"], "%Y-%m-%d")).days
    except:
        days = 7
    sr = serper_search(f"average daily cost {s['destination']} {s['budget_level']}", 4)
    res = call_llm(
        "Travel budget expert. Cost table (flights/hotel/food/transport/activities/misc), per-person/total, savings tips.",
        f"Destination: {s['destination']} | {days} days | Budget: {s['currency']} {s['budget_amount']} for {s['num_travelers']}\nLevel: {s['budget_level']}\nFlights: {s.get('flight_options','')[:300]}\nHotels: {s.get('hotel_options','')[:300]}\nSearch:\n{sr}\nBudget breakdown."
    )
    return {**s, "budget_analysis": res, "current_agent": "itinerary", "progress": 72,
            "messages": [AIMessage(content="Budget done")]}

def itinerary_agent(s):
    print("Building itinerary...")
    try:
        days = (datetime.strptime(s["return_date"], "%Y-%m-%d") - datetime.strptime(s["departure_date"], "%Y-%m-%d")).days
    except:
        days = 7
    ints = ", ".join(s.get("interests", []))
    sr = serper_search(f"best {days} day itinerary {s['destination']} {ints}", 4)
    res = call_llm(
        f"Master itinerary planner. Create {days}-day plan. Each day: morning 8AM-12PM, afternoon 12PM-6PM, evening 6PM+, transport, cost, insider tips.",
        f"Destination: {s['destination']} | {s['departure_date']} to {s['return_date']} ({days} days)\nType: {s['travel_type']} | Interests: {ints}\nResearch: {s.get('destination_research','')[:350]}\nSearch:\n{sr}\n{days}-day itinerary."
    )
    return {**s, "itinerary": res, "current_agent": "booking", "progress": 88,
            "messages": [AIMessage(content=f"{days}-day itinerary done")]}

def booking_agent(s):
    print("Creating checklist...")
    res = call_llm(
        "Travel coordinator. Checklist with checkboxes: 1.Flights 2.Hotels 3.Transfers 4.Activities 5.Insurance 6.Visa 7.Health 8.Packing 9.Emergency contacts 10.48hr pre-departure.",
        f"Trip: {s['destination']} | {s['departure_date']} to {s['return_date']} | {s['num_travelers']} pax | {s['travel_type']}\nSpecial: {s.get('special_requirements','None')}\nBooking checklist with checkboxes."
    )
    return {**s, "booking_checklist": res, "current_agent": "complete", "progress": 100,
            "messages": [AIMessage(content="All agents complete!")]}

print("All 7 agents ready!")

All 7 agents ready!


In [16]:
## CELL 6: Build LangGraph Pipeline
def build_graph():
    g = StateGraph(TravelState)
    g.add_node("destination_research_agent", destination_research_agent)
    g.add_node("flight_search_agent",        flight_search_agent)
    g.add_node("hotel_search_agent",         hotel_search_agent)
    g.add_node("weather_agent",              weather_agent)
    g.add_node("budget_agent",               budget_agent)
    g.add_node("itinerary_agent",            itinerary_agent)
    g.add_node("booking_agent",              booking_agent)
    g.set_entry_point("destination_research_agent")
    g.add_edge("destination_research_agent", "flight_search_agent")
    g.add_edge("flight_search_agent",        "hotel_search_agent")
    g.add_edge("hotel_search_agent",         "weather_agent")
    g.add_edge("weather_agent",              "budget_agent")
    g.add_edge("budget_agent",               "itinerary_agent")
    g.add_edge("itinerary_agent",            "booking_agent")
    g.add_edge("booking_agent",              END)
    return g.compile()

travel_graph = build_graph()
print("LangGraph pipeline compiled!")
print("Flow: Research -> Flights -> Hotels -> Weather -> Budget -> Itinerary -> Bookings -> END")

LangGraph pipeline compiled!
Flow: Research -> Flights -> Hotels -> Weather -> Budget -> Itinerary -> Bookings -> END


In [17]:
## CELL 7: Professional Gradio UI
import gradio as gr
from datetime import datetime, timedelta

today   = datetime.now()
dep_def = (today + timedelta(days=30)).strftime('%Y-%m-%d')
ret_def = (today + timedelta(days=37)).strftime('%Y-%m-%d')

DESTINATIONS = [
    'Paris, France', 'Tokyo, Japan', 'New York, USA', 'Dubai, UAE',
    'Bali, Indonesia', 'London, UK', 'Rome, Italy', 'Barcelona, Spain',
    'Singapore', 'Sydney, Australia', 'Bangkok, Thailand', 'Amsterdam, Netherlands',
    'Maldives', 'Istanbul, Turkey', 'Cape Town, South Africa', 'Cancun, Mexico',
    'Prague, Czech Republic', 'Kyoto, Japan', 'Santorini, Greece', 'Zurich, Switzerland',
]
ORIGINS = [
    'New York, USA', 'London, UK', 'Los Angeles, USA', 'Chicago, USA',
    'Toronto, Canada', 'Sydney, Australia', 'Mumbai, India', 'Dubai, UAE',
    'Singapore', 'Tokyo, Japan', 'Paris, France', 'Frankfurt, Germany',
]
INTERESTS = [
    'Culture & History', 'Food & Cuisine', 'Nature & Outdoors',
    'Shopping', 'Arts & Entertainment', 'Beach & Relaxation',
    'Sports & Adventure', 'Photography', 'Business Networking',
    'Music & Nightlife', 'Wellness & Spa', 'Hiking & Trekking',
    'Wine & Gastronomy', 'Art & Museums', 'Family Activities',
]

# CSS uses hex colors only to avoid parser issues
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@600;700&family=DM+Sans:wght@400;500;600&display=swap');
body, .gradio-container { background:#f8f5f0 !important; font-family:'DM Sans',sans-serif !important; }
.hdr { background:#1a6b5c; border-radius:18px; padding:36px 48px; margin-bottom:22px;
  position:relative; overflow:hidden; box-shadow:0 8px 40px #1a6b5c44; }
.hdr::after { content:'\\2708'; position:absolute; right:48px; top:50%;
  transform:translateY(-50%) rotate(-15deg); font-size:96px; opacity:.10; }
.hdr h1 { font-family:'Playfair Display',serif !important; font-size:2.4em !important;
  color:#ffffff !important; margin:0 0 8px !important; }
.hdr p { color:#d0eeea !important; margin:0; }
.badges { margin-top:14px; display:flex; gap:8px; flex-wrap:wrap; }
.badge { background:#ffffff30; border:1px solid #ffffff44; color:#fff;
  padding:4px 13px; border-radius:50px; font-size:.8em; font-weight:500; }
.pill-row { display:flex; gap:6px; flex-wrap:wrap; align-items:center; margin-bottom:20px; }
.pill { background:#eef7f5; border:1.5px solid #c8e6e3; color:#1a6b5c;
  padding:5px 14px; border-radius:50px; font-size:.82em; font-weight:600; }
.stitle { font-family:'Playfair Display',serif; font-size:1.05em; color:#1a6b5c;
  font-weight:700; margin-bottom:14px; padding-bottom:10px; border-bottom:2px solid #e2d9cf; }
input, select, textarea { border:1.5px solid #e2d9cf !important;
  border-radius:10px !important; background:#fafaf8 !important;
  font-family:'DM Sans',sans-serif !important; }
input:focus, select:focus, textarea:focus {
  border-color:#2a9d8f !important; outline:none !important; }
label { font-weight:500 !important; color:#4a4a5a !important;
  font-size:.85em !important; text-transform:uppercase; letter-spacing:.5px; }
.tabs .tab-nav button { font-family:'DM Sans',sans-serif !important;
  font-weight:500 !important; font-size:.88em !important;
  border-radius:8px 8px 0 0 !important; }
.tabs .tab-nav button.selected { background:#1a6b5c !important;
  color:white !important; font-weight:600 !important; }
"""

def run_planner(orig, dest, ttype, dep, ret, npax, blvl, bamount, curr,
                ints, acc, fcls, sreq):
    if not dest.strip():
        err = 'Please enter a destination.'
        return (err,)*8 + (err,)
    state = {
        'origin':orig, 'destination':dest, 'travel_type':ttype,
        'departure_date':dep, 'return_date':ret, 'num_travelers':int(npax),
        'budget_level':blvl, 'budget_amount':float(bamount), 'currency':curr,
        'interests': ints if isinstance(ints,list) else [ints],
        'accommodation_type':acc, 'flight_class':fcls, 'special_requirements':sreq,
        'destination_research':'', 'flight_options':'', 'hotel_options':'',
        'weather_info':'', 'budget_analysis':'', 'itinerary':'',
        'booking_checklist':'', 'visa_info':'',
        'messages':[HumanMessage(content=f'Plan {orig} to {dest}')],
        'current_agent':'destination_research_agent', 'errors':[], 'progress':0,
    }
    try:
        print(f'\nStarting: {orig} -> {dest} | {dep} to {ret}')
        f = travel_graph.invoke(state)
        return (
            f.get('destination_research','No data'),
            f.get('flight_options','No data'),
            f.get('hotel_options','No data'),
            f.get('weather_info','No data'),
            f.get('budget_analysis','No data'),
            f.get('itinerary','No data'),
            f.get('booking_checklist','No data'),
            f.get('visa_info','No data'),
            f'Complete! {orig} -> {dest} | {dep} - {ret} | {npax} pax | {ttype}',
        )
    except Exception as e:
        err = f'Error: {e}'
        return (err,)*8 + (err,)


HEADER_HTML = (
    '<div class="hdr">'
    '<h1>&#x2708;&#xFE0F; AI Multi-Agent Travel Planner</h1>'
    '<p>LangGraph &middot; Groq LLaMA 3.1-8B-Instant &middot; Serper API &middot; 7 Specialized AI Agents</p>'
    '<div class="badges">'
    '<span class="badge">&#x1F916; 7 AI Agents</span>'
    '<span class="badge">&#x1F50D; Live Search</span>'
    '<span class="badge">&#x26A1; LangGraph</span>'
    '<span class="badge">&#x1F999; LLaMA 3.1-8B</span>'
    '<span class="badge">&#x1F30D; Leisure &amp; Business</span>'
    '</div></div>'
)

PIPE_HTML = (
    '<div class="pill-row">'
    '<span class="pill">&#x1F50D; Research</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x2708;&#xFE0F; Flights</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x1F3E8; Hotels</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x1F324; Weather</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x1F4B0; Budget</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x1F5FA; Itinerary</span><span style="color:#bbb">&#x2192;</span>'
    '<span class="pill">&#x1F4CB; Bookings</span>'
    '</div>'
)

with gr.Blocks(css=CSS, title='AI Travel Planner',
               theme=gr.themes.Soft(primary_hue='teal', neutral_hue='stone')) as demo:

    gr.HTML(HEADER_HTML)
    gr.HTML(PIPE_HTML)

    with gr.Row(equal_height=False):
        with gr.Column(scale=4, min_width=320):
            gr.HTML('<div class="stitle">&#x1F30D; Trip Details</div>')
            with gr.Row():
                orig_dd = gr.Dropdown(choices=ORIGINS, value='New York, USA',
                    label='Origin City', allow_custom_value=True, scale=1)
                dest_dd = gr.Dropdown(choices=DESTINATIONS, value='Paris, France',
                    label='Destination', allow_custom_value=True, scale=1)
            travel_type = gr.Radio(
                choices=['Leisure','Business','Bleisure','Adventure','Family','Honeymoon'],
                value='Leisure', label='Travel Purpose')
            with gr.Row():
                dep_in = gr.Textbox(value=dep_def, label='Departure Date (YYYY-MM-DD)', scale=1)
                ret_in = gr.Textbox(value=ret_def, label='Return Date (YYYY-MM-DD)',    scale=1)
            num_pax = gr.Slider(minimum=1, maximum=20, value=2, step=1, label='Number of Travelers')
            gr.HTML('<div class="stitle" style="margin-top:20px;">&#x1F4B0; Budget</div>')
            with gr.Row():
                blvl = gr.Dropdown(
                    choices=['Budget (Economy)','Mid-Range','Luxury','Ultra-Luxury'],
                    value='Mid-Range', label='Budget Level', scale=1)
                curr = gr.Dropdown(
                    choices=['USD','EUR','GBP','JPY','AUD','CAD','INR','SGD','AED','CHF'],
                    value='USD', label='Currency', scale=1)
            bamount = gr.Slider(minimum=500, maximum=50000, value=5000, step=500, label='Total Budget Amount')
            gr.HTML('<div class="stitle" style="margin-top:20px;">&#x2708;&#xFE0F; Transport and Stay</div>')
            with gr.Row():
                fcls = gr.Dropdown(
                    choices=['Economy','Premium Economy','Business','First Class'],
                    value='Economy', label='Flight Class', scale=1)
                acc = gr.Dropdown(
                    choices=['Hotel','Boutique Hotel','Resort','Hostel','Airbnb / Apartment','Villa','Business Hotel'],
                    value='Hotel', label='Accommodation Type', scale=1)
            gr.HTML('<div class="stitle" style="margin-top:20px;">&#x1F3AF; Interests</div>')
            ints = gr.CheckboxGroup(choices=INTERESTS,
                value=['Culture & History','Food & Cuisine'],
                label='Select your interests (multi-select)')
            gr.HTML('<div class="stitle" style="margin-top:20px;">&#x1F4DD; Notes</div>')
            sreq = gr.Textbox(lines=3,
                placeholder='e.g. vegetarian meals, wheelchair access, anniversary trip, business meetings Day 2-3...',
                label='Special Requirements')
            gr.HTML('<br>')
            sub_btn = gr.Button('Plan My Perfect Trip', variant='primary', size='lg')
            clr_btn = gr.Button('Reset Form', variant='secondary', size='sm')

        with gr.Column(scale=6):
            status = gr.Textbox(
                value='Configure your trip on the left, then click Plan My Perfect Trip',
                label='Agent Status', interactive=False)
            with gr.Tabs():
                with gr.Tab('Research'):
                    o1 = gr.Markdown('*Destination research will appear here...*')
                with gr.Tab('Flights'):
                    o2 = gr.Markdown('*Flight options will appear here...*')
                with gr.Tab('Hotels'):
                    o3 = gr.Markdown('*Hotel recommendations will appear here...*')
                with gr.Tab('Weather'):
                    o4 = gr.Markdown('*Weather forecast will appear here...*')
                with gr.Tab('Budget'):
                    o5 = gr.Markdown('*Budget breakdown will appear here...*')
                with gr.Tab('Itinerary'):
                    o6 = gr.Markdown('*Day-by-day itinerary will appear here...*')
                with gr.Tab('Bookings'):
                    o7 = gr.Markdown('*Booking checklist will appear here...*')
                with gr.Tab('Visa'):
                    o8 = gr.Markdown('*Visa requirements will appear here...*')

    all_in  = [orig_dd, dest_dd, travel_type, dep_in, ret_in, num_pax,
               blvl, bamount, curr, ints, acc, fcls, sreq]
    all_out = [o1, o2, o3, o4, o5, o6, o7, o8, status]

    sub_btn.click(fn=run_planner, inputs=all_in, outputs=all_out, show_progress=True)
    clr_btn.click(
        fn=lambda: (
            'New York, USA', 'Paris, France', 'Leisure',
            dep_def, ret_def, 2, 'Mid-Range', 5000, 'USD',
            ['Culture & History','Food & Cuisine'], 'Hotel', 'Economy', '',
            '*Destination research will appear here...*',
            '*Flight options will appear here...*',
            '*Hotel recommendations will appear here...*',
            '*Weather forecast will appear here...*',
            '*Budget breakdown will appear here...*',
            '*Day-by-day itinerary will appear here...*',
            '*Booking checklist will appear here...*',
            '*Visa requirements will appear here...*',
            'Configure your trip on the left, then click Plan My Perfect Trip'
        ),
        outputs=all_in + all_out
    )

print('Gradio UI ready! Run Cell 8 to launch.')


/tmp/ipykernel_2554/1647381037.py:123: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='AI Travel Planner',
/tmp/ipykernel_2554/1647381037.py:123: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, title='AI Travel Planner',


Gradio UI ready! Run Cell 8 to launch.


In [19]:
## CELL 8: Launch
print('Launching AI Multi-Agent Travel Planner...')
demo.launch(
    share=True,
    debug=False,
    show_error=True,
    inline=True
)

Launching AI Multi-Agent Travel Planner...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9a2a894ca41ef1a134.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## Quick Start Guide

### API Keys
| Service | URL | Free Tier |
|---------|-----|-----------|
| Groq | https://console.groq.com | Free, ultra-fast |
| Serper | https://serper.dev | 2,500 free searches/month |

### Steps
1. Cell 1 — Install packages
2. Cell 2 — Add your API keys
3. Cells 3-7 — Initialize (run all)
4. Cell 8 — Launch app, open the share URL

### Agent Pipeline
```
Research -> Flights -> Hotels -> Weather -> Budget -> Itinerary -> Bookings
```

### Features
- 20+ destination choices via dropdown
- 6 travel types: Leisure, Business, Bleisure, Adventure, Family, Honeymoon
- 15 interest categories with multi-select checkboxes
- 4 budget levels and 10 currency options
- 4 flight class options and 7 accommodation types
- 8 result tabs, one per agent output
